# 02 - Fetch Researchr PC Members

In this notebook, I fetch the visible PC / review committee pages from the
conference websites.

This is different from HotCRP. HotCRP tells me who was in the review system.
The Researchr/SIGPLAN pages tell me who was visible on the conference
website.


## 1 - Setup

In [1]:
import re
import requests

import pandas as pd

from bs4 import BeautifulSoup
from datetime import date
from pathlib import Path
from urllib.parse import urljoin


In [2]:
import sys
from pathlib import Path

for candidate in [Path.cwd(), *Path.cwd().parents]:
    if (candidate / "project_setup.py").exists() and (candidate / "config" / "project_config.yaml").exists():
        if str(candidate) not in sys.path:
            sys.path.insert(0, str(candidate))
        break
else:
    raise RuntimeError(
        "Could not find the repository root. Launch Jupyter from the repo root "
        "or set PYTHONPATH to the folder containing project_setup.py."
    )

from project_setup import setup_project

setup = setup_project()
project_folder = setup.project_folder
PROJECT = project_folder
repo = project_folder
config_path = setup.config_path
project_config = setup.project_config

run_mode = setup.run_mode
inputs_config = setup.inputs
outputs_config = setup.outputs
openalex_config = setup.openalex

allow_network = setup.allow_network
use_existing_data = setup.use_existing_data
overwrite_data = setup.overwrite_data
overwrite_artifacts = setup.overwrite_artifacts
openalex_sample_limit = setup.openalex_sample_limit
openalex_sample_include_work_ids = setup.openalex_sample_include_work_ids

step_1_data_dir = project_folder / "step_1_data"
step_1_artifacts_dir = project_folder / "step_1_artifacts"
raw_dir = step_1_data_dir / "raw"
intermediate_dir = step_1_data_dir / "intermediate"
prepared_dir = step_1_data_dir / "prepared"
summary_tables_dir = step_1_artifacts_dir / "summary_tables"
dependency_tables_dir = step_1_artifacts_dir / "dependency_tables"
check_tables_dir = step_1_artifacts_dir / "check_tables"

raw_dir.mkdir(parents=True, exist_ok=True)
intermediate_dir.mkdir(parents=True, exist_ok=True)
prepared_dir.mkdir(parents=True, exist_ok=True)
summary_tables_dir.mkdir(parents=True, exist_ok=True)
dependency_tables_dir.mkdir(parents=True, exist_ok=True)
check_tables_dir.mkdir(parents=True, exist_ok=True)

TODAY = date.today().isoformat()
RUN_FROM_CACHE = use_existing_data

print(project_folder)
print(f"Run mode: {run_mode}")


/Users/endersari/2026-02-citations-vs-pc-memberships
Run mode: fast


## 2 - PC URLs

In [3]:
urls_pc = {
    "ICFP": {
        2017: "https://icfp17.sigplan.org/committee/icfp-2017-papers-program-committee",
        2018: "https://icfp18.sigplan.org/committee/icfp-2018-papers-program-committee",
        2019: "https://icfp19.sigplan.org/committee/icfp-2019-papers-program-committee",
        2020: "https://icfp20.sigplan.org/committee/icfp-2020-papers-program-committee",
        2021: "https://icfp21.sigplan.org/committee/icfp-2021-papers-program-committee",
        2022: "https://icfp22.sigplan.org/committee/icfp-2022-papers-program-committee",
        2023: "https://icfp23.sigplan.org/committee/icfp-2023-papers-program-committee",
        2024: "https://icfp24.sigplan.org/committee/icfp-2024-papers-icfp-papers-and-events",
        2025: "https://icfp25.sigplan.org/committee/icfp-2025-papers-icfp-papers-and-events",
    },
    "POPL": {
        2017: "https://popl17.sigplan.org/committee/popl-2017-papers-program-committee",
        2018: "https://popl18.sigplan.org/committee/popl-2018-papers-program-committee",
        2019: "https://popl19.sigplan.org/committee/popl-2019-research-papers-program-committee",
        2020: "https://popl20.sigplan.org/committee/popl-2020-papers-program-committee",
        2021: "https://popl21.sigplan.org/committee/POPL-2021-research-papers-program-committee",
        2022: "https://popl22.sigplan.org/committee/POPL-2022-popl-research-papers-program-committee",
        2023: "https://popl23.sigplan.org/committee/POPL-2023-popl-research-papers-program-committee",
        2024: "https://popl24.sigplan.org/committee/POPL-2024-popl-research-papers-program-committee",
        2025: "https://popl25.sigplan.org/committee/POPL-2025-popl-research-papers-program-committee",
    },
    "OOPSLA": {
        2017: "https://2017.splashcon.org/committee/splash-2017-oopsla-program-committee",
        2018: "https://2018.splashcon.org/committee/splash-2018-oopsla-committee",
        2019: "https://2019.splashcon.org/committee/splash-2019-oopsla-review-committee",
        2020: "https://2020.splashcon.org/committee/splash-2020-oopsla-review-committee",
        2021: "https://2021.splashcon.org/committee/splash-2021-oopsla-review-committee",
        2022: "https://2022.splashcon.org/committee/splash-2022-psla-review-committee",
        2023: "https://2023.splashcon.org/committee/splash-2023-oopsla-review-committee",
        2024: "https://2024.splashcon.org/committee/splash-2024-papers-review-committee",
        2025: "https://2025.splashcon.org/committee/splash-2025-OOPSLA-oopsla-review-committee",
    },
    "PLDI": {
        2017: "https://conf.researchr.org/committee/pldi-2017/pldi-2017-program-committee",
        2018: "https://conf.researchr.org/committee/pldi-2018/pldi-2018-program-committee",
        2019: "https://conf.researchr.org/committee/pldi-2019/pldi-2019-papers-program-committee",
        2020: "https://conf.researchr.org/committee/pldi-2020/pldi-2020-papers-program-committee",
        2021: "https://conf.researchr.org/committee/pldi-2021/pldi-2021-papers-program-committee",
        2022: "https://conf.researchr.org/committee/pldi-2022/pldi-2022-pldi-program-committee",
        2023: "https://pldi23.sigplan.org/committee/pldi-2023-pldi-review-committee",
        2024: "https://pldi24.sigplan.org/committee/pldi-2024-papers-pldi-review-committee",
        2025: "https://pldi25.sigplan.org/committee/pldi-2025-papers-pldi-review-committee",
    },
}


In [4]:
url_rows = []

for conf, year_to_url in urls_pc.items():
    for year, url in year_to_url.items():
        url_rows.append({
            "conference": conf,
            "year": year,
            "url": url,
        })

url_df = pd.DataFrame(url_rows).sort_values(["conference", "year"])
print(url_df.shape)
display(url_df)


(36, 3)


,conference,year,url
0,ICFP,2017,https://icfp17.sigplan.org/committee/icfp-2017...
1,ICFP,2018,https://icfp18.sigplan.org/committee/icfp-2018...
2,ICFP,2019,https://icfp19.sigplan.org/committee/icfp-2019...
3,ICFP,2020,https://icfp20.sigplan.org/committee/icfp-2020...
4,ICFP,2021,https://icfp21.sigplan.org/committee/icfp-2021...
5,ICFP,2022,https://icfp22.sigplan.org/committee/icfp-2022...
6,ICFP,2023,https://icfp23.sigplan.org/committee/icfp-2023...
7,ICFP,2024,https://icfp24.sigplan.org/committee/icfp-2024...
8,ICFP,2025,https://icfp25.sigplan.org/committee/icfp-2025...
18,OOPSLA,2017,https://2017.splashcon.org/committee/splash-20...


## 3 - Parser

In [5]:
def parse_researchr_pc(html, page_url):
    soup = BeautifulSoup(html, "html.parser")

    rows = []

    for a in soup.select('a[href*="/profile/"]'):
        h3 = a.select_one("h3.media-heading")
        if h3 is None:
            continue

        role_tags = h3.find_all("small", recursive=False)
        role_tag = role_tags[-1] if len(role_tags) else None
        if role_tag is None:
            role = "PC Member"
        else:
            role_text = role_tag.get_text(" ", strip=True).lower()
            if "associate" in role_text and "chair" in role_text:
                role = "Associate Chair"
            elif "chair" in role_text:
                role = "PC Chair"
            else:
                role = "PC Member"
            role_tag.extract()

        for marker in h3.select("sup"):
            marker.extract()

        name = h3.get_text(" ", strip=True)
        name = re.sub(r"\s+", " ", name).strip()
        if name == "":
            continue

        affiliation_tag = a.select_one("h4 span.text-black")
        if affiliation_tag is None:
            affiliation = ""
        else:
            affiliation = affiliation_tag.get_text(" ", strip=True)
            affiliation = re.sub(r"\s+", " ", affiliation)

        country_tags = a.select("h4 small")
        if len(country_tags) == 0:
            country = ""
        else:
            country = country_tags[-1].get_text(" ", strip=True)
            country = re.sub(r"\s+", " ", country)

        href = a.get("href", "")
        person_url = urljoin(page_url, href)
        researchr_id = person_url.rstrip("/").split("/profile/")[-1]

        rows.append({
            "name": name,
            "role": role,
            "affiliation": affiliation,
            "country": country,
            "person_url": person_url,
            "researchr_id": researchr_id,
        })

    return pd.DataFrame(rows)


## 4 - Fetch Pages

In [6]:
def cache_path(conf, year):
    folder = raw_dir / conf.lower() / "researchr_pc_htmls"
    folder.mkdir(parents=True, exist_ok=True)
    return folder / f"{conf.lower()}{year}_pc_{TODAY}.html"


def latest_cached_page(conf, year):
    folder = raw_dir / conf.lower() / "researchr_pc_htmls"
    files = sorted(folder.glob(f"{conf.lower()}{year}_pc_*.html"))
    if len(files) == 0:
        return None
    return files[-1]


def get_html(conf, year, url):
    if RUN_FROM_CACHE:
        cached = latest_cached_page(conf, year)
        if cached is not None:
            html = cached.read_text(encoding="utf-8", errors="replace")
            return html, "cache", str(cached), url

    response = requests.get(
        url,
        timeout=20,
        headers={"User-Agent": "Mozilla/5.0"},
    )
    response.raise_for_status()

    path = cache_path(conf, year)
    path.write_text(response.text, encoding="utf-8")

    return response.text, "fetched", str(path), response.url


In [7]:
all_members = []
summary_rows = []

for row in url_df.itertuples(index=False):
    conf = row.conference
    year = int(row.year)
    url = row.url

    print(f"{conf} {year}...", end=" ")

    try:
        html, status, path, final_url = get_html(conf, year, url)
        df_year = parse_researchr_pc(html, final_url)
        error = ""
        print(f"{status}, {len(df_year)} rows")
    except Exception as e:
        df_year = pd.DataFrame()
        status = "error"
        path = ""
        final_url = url
        error = repr(e)
        print("error")

    if len(df_year) > 0:
        df_year["conference"] = conf
        df_year["year"] = year
        df_year["source"] = "Researchr PC"
        df_year["source_url"] = url
        df_year["final_url"] = final_url
        all_members.append(df_year)

    summary_rows.append({
        "conference": conf,
        "year": year,
        "url": url,
        "final_url": final_url,
        "status": status,
        "cache_path": path,
        "n_members": len(df_year),
        "error": error,
    })

pc_members = pd.concat(all_members, ignore_index=True)
pc_summary = pd.DataFrame(summary_rows)


ICFP 2017... fetched, 23 rows
ICFP 2018... fetched, 18 rows
ICFP 2019... fetched, 20 rows
ICFP 2020... fetched, 17 rows
ICFP 2021... fetched, 30 rows
ICFP 2022... fetched, 42 rows
ICFP 2023... fetched, 54 rows
ICFP 2024... fetched, 51 rows
ICFP 2025... fetched, 63 rows
OOPSLA 2017... fetched, 31 rows
OOPSLA 2018... fetched, 30 rows
OOPSLA 2019... fetched, 32 rows
OOPSLA 2020... fetched, 32 rows
OOPSLA 2021... fetched, 51 rows
OOPSLA 2022... fetched, 47 rows
OOPSLA 2023... fetched, 78 rows
OOPSLA 2024... fetched, 108 rows
OOPSLA 2025... fetched, 115 rows
PLDI 2017... fetched, 36 rows
PLDI 2018... fetched, 38 rows
PLDI 2019... fetched, 39 rows
PLDI 2020... fetched, 43 rows
PLDI 2021... fetched, 126 rows
PLDI 2022... fetched, 119 rows
PLDI 2023... fetched, 113 rows
PLDI 2024... fetched, 135 rows
PLDI 2025... fetched, 137 rows
POPL 2017... fetched, 29 rows
POPL 2018... fetched, 52 rows
POPL 2019... fetched, 52 rows
POPL 2020... fetched, 54 rows
POPL 2021... fetched, 52 rows
POPL 2022... fe

## 5 - Save Data

In [8]:
RESEARCHR_ID_MAP = {
    "bengreenman1": "bengreenman",
    "constantinenea1": "constantinenea",
    "filipsieczkowski1": "filipsieczkowski",
    "gerwinklein1": "gerwinklein",
    "kewang4": "kewang",
    "laurenpick1": "laurenpick",
    "nobukoyoshida1": "nobukoyoshida",
    "ronghuigu1": "ronghuigu",
    "samanamarsinghe": "samanamarasinghe",
    "shazqadeer1": "shazqadeer",
    "stephenkell2": "stephenkell",
    "tylersorensen1": "tylersorensen",
    "yudavidliu1": "yudavidliu",
}

pc_members["canonical_researchr_id"] = (
    pc_members["researchr_id"].replace(RESEARCHR_ID_MAP)
)

canonical_map_rows = []
for old_id, canonical_id in RESEARCHR_ID_MAP.items():
    rows = pc_members.query("researchr_id == @old_id")
    if len(rows) == 0:
        continue
    canonical_map_rows.append({
        "researchr_id": old_id,
        "canonical_researchr_id": canonical_id,
        "name": " | ".join(sorted(rows["name"].dropna().unique())),
        "n_rows": len(rows),
        "conferences": ", ".join(sorted(rows["conference"].dropna().unique())),
    })

canonical_id_map = pd.DataFrame(canonical_map_rows)

pc_members = pc_members[
    [
        "conference", "year", "source", "name", "role", "affiliation",
        "country", "person_url", "researchr_id", "canonical_researchr_id",
        "source_url", "final_url",
    ]
].sort_values(["conference", "year", "name"]).reset_index(drop=True)

pc_summary = pc_summary.sort_values(["conference", "year"]).reset_index(drop=True)

pc_members.to_parquet(intermediate_dir / "researchr_pc_members.parquet", index=False)
pc_summary.to_csv(dependency_tables_dir / "researchr_pc_fetch_summary.csv", index=False)
canonical_id_map.to_csv(dependency_tables_dir / "researchr_pc_canonical_id_map.csv", index=False)

print(pc_members.shape)
display(pc_members.head())


(2180, 12)


,conference,year,source,name,role,affiliation,country,person_url,researchr_id,canonical_researchr_id,source_url,final_url
0,ICFP,2017,Researchr PC,Adam Chlipala,PC Member,"Massachusetts Institute of Technology, USA",United States,https://icfp17.sigplan.org/profile/adamchlipala,adamchlipala,adamchlipala,https://icfp17.sigplan.org/committee/icfp-2017...,https://icfp17.sigplan.org/committee/icfp-2017...
1,ICFP,2017,Researchr PC,Alan Jeffrey,PC Member,Mozilla Research,United States,https://icfp17.sigplan.org/profile/alanjeffrey1,alanjeffrey1,alanjeffrey1,https://icfp17.sigplan.org/committee/icfp-2017...,https://icfp17.sigplan.org/committee/icfp-2017...
2,ICFP,2017,Researchr PC,Alexandra Silva,PC Member,University College London,United Kingdom,https://icfp17.sigplan.org/profile/alexandrasilva,alexandrasilva,alexandrasilva,https://icfp17.sigplan.org/committee/icfp-2017...,https://icfp17.sigplan.org/committee/icfp-2017...
3,ICFP,2017,Researchr PC,Ben Lippmeier,PC Member,Digital Asset / UNSW Australia,,https://icfp17.sigplan.org/profile/benlippmeier,benlippmeier,benlippmeier,https://icfp17.sigplan.org/committee/icfp-2017...,https://icfp17.sigplan.org/committee/icfp-2017...
4,ICFP,2017,Researchr PC,Beta Ziliani,PC Member,"FAMAF, UNC and CONICET",Argentina,https://icfp17.sigplan.org/profile/betaziliani,betaziliani,betaziliani,https://icfp17.sigplan.org/committee/icfp-2017...,https://icfp17.sigplan.org/committee/icfp-2017...


## 6 - Quick Checks

In [9]:
count_table = pc_summary.pivot_table(
    index="year",
    columns="conference",
    values="n_members",
    aggfunc="sum",
).astype(int)

display(count_table)


conference,ICFP,OOPSLA,PLDI,POPL
year,,,,
2017,23,31,36,29
2018,18,30,38,52
2019,20,32,39,52
2020,17,32,43,54
2021,30,51,126,52
2022,42,47,119,56
2023,54,78,113,84
2024,51,108,135,91
2025,63,115,137,82


In [10]:
print("Rows by conference")
print(pc_members.groupby("conference").size())

print("\nUnique researchr_id by conference")
print(pc_members.groupby("conference")["researchr_id"].nunique())

print("\nUnique canonical_researchr_id by conference")
print(pc_members.groupby("conference")["canonical_researchr_id"].nunique())

print("\nOverall unique identifiers")
print("name:", pc_members["name"].nunique())
print("researchr_id:", pc_members["researchr_id"].nunique())
print("canonical_researchr_id:", pc_members["canonical_researchr_id"].nunique())

print("\nFetch status")
print(pc_summary["status"].value_counts())


Rows by conference
conference
ICFP      318
OOPSLA    524
PLDI      786
POPL      552
dtype: int64

Unique researchr_id by conference
conference
ICFP      250
OOPSLA    386
PLDI      491
POPL      384
Name: researchr_id, dtype: int64

Unique canonical_researchr_id by conference
conference
ICFP      250
OOPSLA    383
PLDI      484
POPL      384
Name: canonical_researchr_id, dtype: int64

Overall unique identifiers
name: 958
researchr_id: 971
canonical_researchr_id: 958

Fetch status
status
fetched    36
Name: count, dtype: int64


## 7 - What I learned

This table is the visible PC / review committee data from the conference
websites. This is the source I will compare against HotCRP in the next step.
